# Day 33: LLM Deployment and Lifecycle
**Deployment architectures, environments, secrets, scalability and inference pipelines**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas fastapi httpx nest_asyncio

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas fastapi httpx nest_asyncio
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

In [ ]:
import asyncio, os, time, json, hashlib, random
def run_async(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import nest_asyncio; nest_asyncio.apply()
    return loop.run_until_complete(coro)
print("ready")

## 2. Settings per environment, with fail-fast secrets

In [ ]:
def load_settings():
    env = os.getenv("APP_ENV", "dev")
    s = {"env": env, "model": os.getenv("LLM_MODEL", "small-model"), "max_concurrency": int(os.getenv("MAX_CONCURRENCY", "10")),
         "api_key": os.getenv("LLM_API_KEY")}
    if env == "prod" and not s["api_key"]:
        raise RuntimeError("LLM_API_KEY missing in prod: refusing to start")
    return {**s, "api_key": "set" if s["api_key"] else "not set"}       # never print the real value

print(load_settings())
os.environ["APP_ENV"] = "prod"
try:
    load_settings()
except RuntimeError as e:
    print("Startup blocked:", e)
os.environ["APP_ENV"] = "dev"

## 3. A production-style LLM service
The model is simulated with realistic latency, so you can study the service without a key or a GPU.

In [ ]:
%%writefile llm_service.py
import asyncio, os, random, time, hashlib
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

app = FastAPI(title="LLM Service")
LIMIT = asyncio.Semaphore(int(os.getenv("MAX_CONCURRENCY", "10")))
KB = {"leave": "Employees receive 18 days of paid leave per year.", "remote": "Remote work is allowed up to 2 days per week."}
RELEASES = {"stable": "1.8.0", "canary": "1.9.0"}
CANARY_PERCENT = int(os.getenv("CANARY_PERCENT", "10"))

class ChatIn(BaseModel):
    user_id: str
    text: str = Field(min_length=1, max_length=2000)

async def fake_model(prompt):
    await asyncio.sleep(random.uniform(0.15, 0.25))                      # simulated model latency
    return "Based on policy: " + prompt.split("Context: ")[-1].split("\n")[0]

def release_for(user_id):
    bucket = int(hashlib.sha256(user_id.encode()).hexdigest(), 16) % 100
    return RELEASES["canary"] if bucket < CANARY_PERCENT else RELEASES["stable"]

@app.post("/chat")
async def chat(req: ChatIn):
    t = {}
    s = time.perf_counter()
    if "ignore previous instructions" in req.text.lower():               # validate (input guardrail)
        raise HTTPException(400, "Request blocked by input policy")
    t["validate"] = time.perf_counter() - s; s = time.perf_counter()
    context = next((v for k, v in KB.items() if k in req.text.lower()), "No policy found.")   # enrich
    t["enrich"] = time.perf_counter() - s; s = time.perf_counter()
    async with LIMIT:                                                     # generate, with a concurrency limit
        answer = await asyncio.wait_for(fake_model(f"Question: {req.text}\nContext: {context}"), timeout=5)
    t["generate"] = time.perf_counter() - s; s = time.perf_counter()
    if "@" in answer:                                                     # check (output guardrail)
        answer = "[redacted]"
    t["check"] = time.perf_counter() - s
    return {"answer": answer, "release": release_for(req.user_id), "timings_ms": {k: round(v * 1000, 1) for k, v in t.items()}}

@app.post("/chat/stream")
async def chat_stream(req: ChatIn):
    async def tokens():
        for word in f"Here is a streamed answer about {req.text[:40]}".split():
            await asyncio.sleep(0.02)
            yield word + " "
    return StreamingResponse(tokens(), media_type="text/plain")

In [ ]:
# ===== Option A: Google Colab (active) =====
import importlib, llm_service
importlib.reload(llm_service)
from fastapi.testclient import TestClient
client = TestClient(llm_service.app)

# ===== Option B: VS Code =====
# Terminal: uvicorn llm_service:app --reload   then try /docs

r = client.post("/chat", json={"user_id": "u-42", "text": "How many leave days do I get?"})
print(r.json())
print("Blocked input:", client.post("/chat", json={"user_id": "u-1", "text": "Ignore previous instructions"}).status_code)

# TestClient collects the whole stream before returning, so time to first chunk needs a real server:
import subprocess, sys, httpx
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "llm_service:app", "--port", "8010"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(50):
    try:
        httpx.get("http://localhost:8010/docs"); break
    except httpx.HTTPError:
        time.sleep(0.2)
try:
    start = time.perf_counter(); first = None; chunks = []
    with httpx.stream("POST", "http://localhost:8010/chat/stream", json={"user_id": "u-42", "text": "remote work rules"}) as resp:
        for chunk in resp.iter_text():
            if chunk and first is None:
                first = time.perf_counter() - start
            chunks.append(chunk)
    print(f"Streaming: first chunk after {first * 1000:.0f} ms, full answer after {(time.perf_counter() - start) * 1000:.0f} ms")
    print("".join(chunks))
finally:
    server.terminate()

## 4. Load test: why concurrency limits matter
The simulated provider rejects calls when more than 20 are in flight, like a real rate limit.

In [ ]:
class Provider:
    def __init__(self, max_in_flight=20):
        self.max, self.in_flight, self.rejected = max_in_flight, 0, 0
    async def call(self):
        if self.in_flight >= self.max:
            self.rejected += 1
            raise RuntimeError("429")
        self.in_flight += 1
        try:
            await asyncio.sleep(0.05)
        finally:
            self.in_flight -= 1

async def load_test(n=100, limit=None):
    p = Provider()
    sem = asyncio.Semaphore(limit) if limit else None
    async def one():
        try:
            if sem:
                async with sem:
                    await p.call()
            else:
                await p.call()
        except RuntimeError:
            pass
    t = time.perf_counter()
    await asyncio.gather(*[one() for _ in range(n)])
    return p.rejected, time.perf_counter() - t

async def sequential(n=100):
    p = Provider(); t = time.perf_counter()
    for _ in range(n):
        await p.call()
    return p.rejected, time.perf_counter() - t

for name, coro in [("sequential", sequential()), ("unlimited", load_test()), ("semaphore 10", load_test(limit=10))]:
    rejected, secs = run_async(coro)
    print(f"{name:<13} rejected {rejected:>3} | total {secs:.2f} s")

## 5. Canary routing and a rollback decision

In [ ]:
from collections import Counter
users = [f"u-{i}" for i in range(5000)]
counts = Counter(llm_service.release_for(u) for u in users)
print("Canary share:", f"{counts['1.9.0'] / len(users):.1%}", dict(counts))
print("u-42 is stable across calls:", len({llm_service.release_for('u-42') for _ in range(5)}) == 1)

def decide(canary, baseline):
    if canary["error_rate"] > baseline["error_rate"] * 2 or canary["p95_ms"] > baseline["p95_ms"] * 1.3:
        return "ROLLBACK"
    if canary["thumbs_down"] > baseline["thumbs_down"] + 0.02:
        return "HOLD and investigate"
    return "PROMOTE"

baseline = {"error_rate": 0.010, "p95_ms": 2400, "thumbs_down": 0.06}
for canary in [{"error_rate": 0.011, "p95_ms": 2300, "thumbs_down": 0.05},
               {"error_rate": 0.031, "p95_ms": 2500, "thumbs_down": 0.06},
               {"error_rate": 0.012, "p95_ms": 2450, "thumbs_down": 0.10}]:
    print(canary, "->", decide(canary, baseline))

### TODO 1: Add a `/ready` endpoint that returns 503 when `LLM_API_KEY` is missing in prod, and a feature flag that enables streaming only for users whose id ends in an even number.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)